# Torchvision Transforms: Image Preprocessing

Reach for this when you need: 
- Reference for modern `torchvision.transforms.v2` API.
- To implement data augmentation for training stabilization.
- To transform raw PIL/Numpy images into PyTorch-ready tensors.

In [1]:
import torch
from torchvision.transforms import v2
from PIL import Image
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 1. Modern v2 Transforms

| Transform | Description | Usage |
| :--- | :--- | :--- |
| `Resize` | Change spatial dimensions | Standardizing input resolutions |
| `RandomResizedCrop` | Crop and resize | SOTA augmentation for ImageNet training |
| `ColorJitter` | Randomize brightness/contrast | Robustness to lighting conditions |
| `ToDtype` | Cast and scale (0-1) | Modern successor to `ToTensor()` |

In [2]:
# Recommended Pipeline for Training
train_tforms = v2.Compose([
    v2.RandomResizedCrop(size=(224, 224), antialias=True),
    v2.RandomHorizontalFlip(p=0.5),
    v2.ToImage(), 
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Recommended Pipeline for Inference
test_tforms = v2.Compose([
    v2.Resize(256, antialias=True),
    v2.CenterCrop(224),
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

## 2. Advanced Augmentation (MixUp / CutMix)

Modern COMPUTER VISION techniques that interpolate between multiple images to improve generalization.

✅ **Use when**: Training large models (ViT, ResNet-101) on large datasets like ImageNet.
❌ **Don't use when**: Training on very small datasets where synthetic samples might confuse the model.

In [ ]:
cutmix = v2.CutMix(num_classes=1000)
mixup = v2.MixUp(num_classes=1000)

# Apply to BATCHES inside the training loop
# inputs, labels = cutmix(inputs, labels)

### Common Pitfalls
- **Antialiasing**: In `v2.Resize()`, always set `antialias=True` to match PIL's behavior and avoid artifacts.
- **Normalization Order**: Always apply `Normalize` AFTER `ToDtype` if you want it to act on [0, 1] scaled floats.
- **Inplace**: Be careful with inplace transforms on shared tensors in DataLoaders.

### Key Takeaways
- `v2` transforms are faster and handle Bounding Boxes/Segmentation Masks automatically.
- Always use ImageNet mean/std for models pretrained on ImageNet (ResNet, MobileNet).
- `Compose` is a sequential wrapper; order matters significantly.